# 09.9 - LLM APIs

**Phase:** 09 - Generative AI

**Status:** VERIFIED

---

## 1. What Are We Solving?

LLM APIs are web services that accept text and return generated responses, abstracting away model hosting, scaling, and infrastructure. Most real applications use APIs rather than running models locally. Understanding authentication, rate limits, error handling, and cost is essential for building production systems.

## 2. Why Does This Matter?

APIs are the practical gateway to real models. Poor key management, missing error handling, or unchecked token usage cause security incidents, outages, and cost overruns.

## 3. Prerequisites

- Unit 09.6 (inference)
- Unit 09.8 (alignment)
- Basic HTTP / API knowledge

## 4. Learning Objectives

- Structure a chat-completion request correctly
- Understand message roles (system, user, assistant)
- Manage API keys safely
- Estimate tokens and cost

## 5. Mental Model

An LLM API is a very powerful autocomplete service: you send messages and parameters, it returns a completion plus usage metadata. You pay per token and must respect rate limits.

```text
your app -> POST {model, messages, temperature, max_tokens} -> LLM service -> {content, usage}
```


## 6. Setup & API Key Safety

**IMPORTANT:** No API key is available in this environment. We show the real client structure but wrap every call in a **mock** that returns a canned response, so the notebook runs offline. In production you would set `OPENAI_API_KEY` and call the real endpoint.


In [1]:
import matplotlib
matplotlib.use('Agg')
import os

# Never hardcode keys. Read from the environment (or a secret manager).
api_key = os.environ.get("OPENAI_API_KEY", "<set-me>")
print("API key present?", api_key != "<set-me>" and bool(api_key))
print("In production: export OPENAI_API_KEY=sk-... before running.")


API key present? True
In production: export OPENAI_API_KEY=sk-... before running.


## 7. A Mock LLM Client

We build an object that mirrors `openai.OpenAI().chat.completions.create(...)` so the calling code is identical to production, but the response is canned. Replace `mock_llm` with the real client when you have a key.


In [2]:
class MockMessage:
    def __init__(self, content):
        self.content = content
        self.finish_reason = "stop"

class MockChoices:
    def __init__(self, content):
        self.message = MockMessage(content)

class MockUsage:
    prompt_tokens = 12
    completion_tokens = 8
    total_tokens = 20

class MockResponse:
    def __init__(self, content):
        self.choices = [MockChoices(content)]
        self.usage = MockUsage()

def mock_llm_completion(messages, **kwargs):
    # In production: return openai_client.chat.completions.create(model=..., messages=..., **kwargs)
    # Requires OPENAI_API_KEY and network access. Here we return a canned response.
    last = messages[-1]["content"]
    content = f"[mock] You asked: {last!r}"
    return MockResponse(content)

resp = mock_llm_completion([{"role": "user", "content": "Hello"}])
print("Response content:", resp.choices[0].message.content)
print(f"Usage: {resp.usage.total_tokens} total tokens")


Response content: [mock] You asked: 'Hello'
Usage: 20 total tokens


## 8. Message Roles

A chat request is a list of messages: `system` (instructions/tone), `user` (input), `assistant` (prior model replies). Getting roles right is key to multi-turn conversation.


In [3]:
messages = [
    {"role": "system",    "content": "You are a helpful assistant."},
    {"role": "user",      "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
    {"role": "user",      "content": "And Germany?"},
]
print(f"Request has {len(messages)} messages:")
for m in messages:
    print(f"  [{m['role']:9s}] {m['content'][:40]}")
print("\nThe system message steers behavior; user/assistant carry the conversation.")


Request has 4 messages:
  [system   ] You are a helpful assistant.
  [user     ] What is the capital of France?
  [assistant] The capital of France is Paris.
  [user     ] And Germany?

The system message steers behavior; user/assistant carry the conversation.


## 9. Request Parameters

`model`, `temperature`, and `max_tokens` are sent with the messages. We show the production call shape wrapped in the mock.


In [4]:
def call_llm(messages, model="gpt-4", temperature=0.7, max_tokens=200):
    # Production body (requires a key):
    # import openai; client = openai.OpenAI();
    # return client.chat.completions.create(
    #     model=model, messages=messages, temperature=temperature, max_tokens=max_tokens)
    resp = mock_llm_completion(messages, model=model, temperature=temperature, max_tokens=max_tokens)
    return resp

out = call_llm([{"role": "user", "content": "Explain Python in one sentence."}], max_tokens=50)
print("Model reply:", out.choices[0].message.content)
print("finish_reason:", out.choices[0].message.finish_reason)


Model reply: [mock] You asked: 'Explain Python in one sentence.'
finish_reason: stop


## 10. System Prompt for Behavior Control

A system prompt changes tone/role without changing the question. We compare two system prompts.


In [5]:
def system_prompt_demo(system, user):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    return call_llm(messages, temperature=0.8)

r1 = system_prompt_demo("You are a pirate.", "What is Python?")
r2 = system_prompt_demo("You are a math professor.", "What is Python?")
print("[pirate system]   ", r1.choices[0].message.content)
print("[professor system]", r2.choices[0].message.content)
print("\nSame question, different system context -> different behavior in a real model.")


[pirate system]    [mock] You asked: 'What is Python?'
[professor system] [mock] You asked: 'What is Python?'

Same question, different system context -> different behavior in a real model.


## 11. Cost & Token Estimation

Billing is per token. Log usage and multiply by price to budget.


In [6]:
def estimate_cost(usage, price_per_1k_in=0.01, price_per_1k_out=0.03):
    cost = usage.prompt_tokens / 1000 * price_per_1k_in + usage.completion_tokens / 1000 * price_per_1k_out
    return cost

resp = call_llm([{"role": "user", "content": "Write a short story."}])
print(f"prompt tokens: {resp.usage.prompt_tokens}")
print(f"completion tokens: {resp.usage.completion_tokens}")
print(f"est. cost: ${estimate_cost(resp.usage):.5f}")
print("\nSet max_tokens to cap cost and latency.")


prompt tokens: 12
completion tokens: 8
est. cost: $0.00036

Set max_tokens to cap cost and latency.


## 12. Error Handling & Debugging

Common API errors and their meaning. We simulate a rate-limit (429) retry pattern.


In [7]:
import time

class SimulatedRateLimit(Exception):
    pass

attempts = [True, True, False]   # two failures then success
def flaky_call():
    if attempts.pop(0):
        raise SimulatedRateLimit("429 Too Many Requests")
    return "success"

for i in range(3):
    try:
        result = flaky_call()
        print(f"Attempt {i}: {result}")
        break
    except SimulatedRateLimit as e:
        wait = 2 ** i          # exponential backoff
        print(f"Attempt {i}: {e} -> retry in {wait}s")
        time.sleep(min(wait, 0.1))
print("\nIn production use exponential backoff; never retry immediately forever.")


Attempt 0: 429 Too Many Requests -> retry in 1s


Attempt 1: 429 Too Many Requests -> retry in 2s


Attempt 2: success



In production use exponential backoff; never retry immediately forever.


## 13. Common API Mistakes & Debugging

| Symptom | Cause | Fix |
|---|---|---|
| 401 Unauthorized | invalid/missing key | check env var / key validity |
| 429 Too Many Requests | rate limit | exponential backoff |
| 500 Server Error | service issue | retry with backoff |
| Unexpected output | wrong model/params | log request params |

## 14. Real-World Considerations

- Never hardcode API keys - use environment variables or secret managers.
- Retry with exponential backoff; add jitter.
- Log token usage for every request to track costs.
- Use the simplest model that meets quality requirements.

## 15. Common Mistakes

- Hardcoding keys in source (gets committed/shared).
- No rate-limit handling.
- No max_tokens (unbounded cost).
- Wrong model for the task.

## 16. When NOT to Use an API

- Privacy/cost control/offline needs -> local model.
- Maximum latency control or full ownership -> self-host.

## 17. Challenge

Build a small chat loop that keeps history and enforces a token budget: stop appending when total tokens exceed a limit.


In [8]:
def chat_loop(initial_messages, budget=60):
    history = list(initial_messages)
    total = 0
    for user_msg in ["Hi", "Tell me about AI", "And deep learning?"]:
        history.append({"role": "user", "content": user_msg})
        resp = call_llm(history, max_tokens=20)
        total += resp.usage.completion_tokens
        history.append({"role": "assistant", "content": resp.choices[0].message.content})
        if total > budget:
            print(f"Budget exceeded ({total} tokens). Stopping.")
            break
    print(f"Final history length: {len(history)}, tokens used: {total}")

chat_loop([{"role": "system", "content": "Be brief."}])
print("\nToken budgeting keeps costs under control.")


Final history length: 7, tokens used: 24

Token budgeting keeps costs under control.


## 18. Closed-Book Recall

1. What are the roles in a chat-completion request?
2. How do you handle a 429 rate-limit error?
3. Why should you set max_tokens?
4. What is the difference between system, user, and assistant messages?

## 19. Teach-Back Questions

Explain to another person:

- How a chat-completion API call is structured.
- How you would secure an API key and control costs.

## 20. Summary

You built a mock LLM client matching the real API shape, used message roles and system prompts, estimated cost, and implemented rate-limit retry logic - all without needing a live API key.

## 21. Further Experiment

- When you obtain a key, swap `mock_llm_completion` for the real OpenAI client and rerun.
- Add jitter to the backoff and monitor total request latency.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: none required at runtime (mock), openai optional
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
